## Test PDF Download on One Real Document

In [1]:
import requests

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3"
}

url = "https://iris.who.int/server/api/core/bitstreams/f062769d-f075-4a00-87af-0a2106e0bd04/content"
response = requests.get(url, headers=headers, timeout=20)

print("Status code:", response.status_code)
print("Content-Type:", response.headers.get("Content-Type"))
print("Actual bytes downloaded:", len(response.content))

Status code: 200
Content-Type: application/pdf;charset=UTF-8
Actual bytes downloaded: 843539


## Resolve an Old Handle to Its New Bitstream URL via DSpace's REST API

In [2]:
handle = "10665/353829"  # the tuberculosis guideline's old handle ( which we search manually in the WHO IRIS database ) 

pid_url = f"https://iris.who.int/server/api/pid/find?id=hdl:{handle}"
response = requests.get(pid_url, headers=headers, timeout=20)

print("Status code:", response.status_code)
print(response.text[:1000]) # we will get uuid from this response, which we can use to get the actual PDF content from the WHO IRIS database.

Status code: 200
{
  "id" : "fd4f105d-7b41-488c-985e-0494067c77ef",
  "uuid" : "fd4f105d-7b41-488c-985e-0494067c77ef",
  "name" : "WHO consolidated guidelines on tuberculosis: module 4: treatment: drug-susceptible tuberculosis treatment",
  "handle" : "10665/353829",
  "metadata" : {
    "dc.contributor.author" : [ {
      "value" : "World Health Organization",
      "language" : "en_US",
      "authority" : null,
      "confidence" : -1,
      "place" : 0
    } ],
    "dc.coverage.spatial" : [ {
      "value" : "Geneva",
      "language" : "en_US",
      "authority" : null,
      "confidence" : -1,
      "place" : 0
    } ],
    "dc.date.accessioned" : [ {
      "value" : "2022-05-04T14:31:25Z",
      "language" : null,
      "authority" : null,
      "confidence" : -1,
      "place" : 0
    } ],
    "dc.date.available" : [ {
      "value" : "2022-05-25T12:00:00Z",
      "language" : null,
      "authority" : null,
      "confidence" : -1,
      "place" : 0
    } ],
    "dc.date.issue

## Get the Item's Bitstreams

In [3]:
import json

item_uuid = "fd4f105d-7b41-488c-985e-0494067c77ef"

bundles_url = f"https://iris.who.int/server/api/core/items/{item_uuid}/bundles"
response = requests.get(bundles_url, headers=headers, timeout=20)

print("Status code:", response.status_code)
data = response.json()
print(json.dumps(data, indent=4)[:1000])  # Print the first 1000 characters of the JSON response for inspection

for bundle in data["_embedded"]["bundles"]:
    print(bundle.get("name"), "->", bundle["_links"]["bitstreams"]["href"]) # Print the name and link to each bitstream in the bundle

Status code: 200
{
    "_embedded": {
        "bundles": [
            {
                "uuid": "d3a065ee-38c5-4cff-b5e2-750cff5a1f4d",
                "name": "ORIGINAL",
                "handle": null,
                "metadata": {
                    "dc.title": [
                        {
                            "value": "ORIGINAL",
                            "language": null,
                            "authority": null,
                            "confidence": -1,
                            "place": 0
                        }
                    ]
                },
                "type": "bundle",
                "_links": {
                    "item": {
                        "href": "https://iris.who.int/server/api/core/bundles/d3a065ee-38c5-4cff-b5e2-750cff5a1f4d/item"
                    },
                    "bitstreams": {
                        "href": "https://iris.who.int/server/api/core/bundles/d3a065ee-38c5-4cff-b5e2-750cff5a1f4d/bitstreams"
            

## Get the ORIGINAL Bundle's Bitstream (the Real PDF)

In [4]:
original_bitstreams_url = "https://iris.who.int/server/api/core/bundles/d3a065ee-38c5-4cff-b5e2-750cff5a1f4d/bitstreams"
response = requests.get(original_bitstreams_url, headers=headers, timeout=20)

data = response.json()
print(json.dumps(data, indent=4)[:2000])  # Print the first 2000 characters of the JSON response for inspection


for bitstream in data["_embedded"]["bitstreams"]:
    print(bitstream["name"])                      # original file name
    print(bitstream["_links"]["content"]["href"]) # original content link
    

{
    "_embedded": {
        "bitstreams": [
            {
                "id": "cf34aa08-c5d4-4b64-85cd-2ec1e6b14d91",
                "uuid": "cf34aa08-c5d4-4b64-85cd-2ec1e6b14d91",
                "name": "9789240048126-eng.pdf",
                "handle": null,
                "metadata": {
                    "dc.description": [
                        {
                            "value": "",
                            "language": null,
                            "authority": null,
                            "confidence": -1,
                            "place": 0
                        }
                    ],
                    "dc.source": [
                        {
                            "value": "9789240048126-eng.pdf",
                            "language": null,
                            "authority": null,
                            "confidence": -1,
                            "place": 0
                        }
                    ],
                    

## Wrap It Into One Reusable Function

In [5]:
def resolve_who_pdf_url(handle: str, headers: dict) -> str:
    """
    Given an old WHO IRIS handle (e.g., '10665/353829'), resolve it through
    DSpace's REST API to find the actual downloadable PDF URL.
    Returns None if resolution fails at any step.
    """
    # Step 1: handle -> uuid
    pid_url = f"https://iris.who.int/server/api/pid/find?id=hdl:{handle}"
    r = requests.get(pid_url, headers=headers, timeout=20)
    if r.status_code != 200:
        return None
    item_uuid = r.json()["uuid"]

    # Step 2: uuid -> original bundle
    bundles_url = f"https://iris.who.int/server/api/core/items/{item_uuid}/bundles"
    r = requests.get(bundles_url, headers=headers, timeout=20)
    if r.status_code != 200:
        return None
    bundles = r.json()["_embedded"]["bundles"]
    original_bundle = next((b for b in bundles if b["name"] == "ORIGINAL"), None)
    if not original_bundle:
        return None

    # Step 3: original bundle -> bitstreams
    bitstreams_url = original_bundle["_links"]["bitstreams"]["href"]
    r = requests.get(bitstreams_url, headers=headers, timeout=20)
    if r.status_code != 200:
        return None
    bitstreams = r.json()["_embedded"]["bitstreams"]
    if not bitstreams:
        return None

    # Take the first bitstream (usually the main PDF)
    return bitstreams[0]["_links"]["content"]["href"]


# Test it on a handle we haven't tried yet - malaria
test_url = resolve_who_pdf_url("10665/373339", headers=headers) # --> handle are been maually searched in the WHO IRIS database, and this is the handle for the malaria guideline.  
print(test_url)

https://iris.who.int/server/api/core/bitstreams/8fa903e7-5502-4c33-a300-7a3f3469bfab/content


## Extract Handles From Our Curated URLs and Resolve Them All

In [6]:
# topic -> handle (extracted from the URLs we found during research)
who_handles = {
    "tuberculosis": "10665/353829",
    "hypertension": "10665/344424",
    "diabetes": None,  # cdn.who.int URL, not an IRIS handle - handle separately
    "obesity": None,   # cdn.who.int URL, not an IRIS handle - handle separately
    "asthma_copd_pen": "10665/334186",
    "cvd_risk": "10665/43685",
    "pneumonia": "10665/137319",
    "covid19": "10665/365580",
    "malaria": "10665/373339",
    "hiv": None,  # already have direct bitstream URL from search
    "hepatitis_b": "10665/154590",
    "hepatitis_c": "10665/366869",
    "dengue": "10665/76887",
    "typhoid": None,  # already have direct bitstreams URL from search
    "mhgap": "10665/204132",
    "malnutrition": "10665/95584",
    "anemia_pregnancy": "10665/376196",
    "breast_cancer": "10665/137339",
}

resolved_urls = {}
for topic, handle in who_handles.items():
    if handle is None:
        print(f"{topic}: SKIPPED (no handle, needs manual URL)")
        continue
    url = resolve_who_pdf_url(handle, headers=headers)
    resolved_urls[topic] = url
    print(f"{topic}: {url}")

tuberculosis: https://iris.who.int/server/api/core/bitstreams/cf34aa08-c5d4-4b64-85cd-2ec1e6b14d91/content
hypertension: https://iris.who.int/server/api/core/bitstreams/f062769d-f075-4a00-87af-0a2106e0bd04/content
diabetes: SKIPPED (no handle, needs manual URL)
obesity: SKIPPED (no handle, needs manual URL)
asthma_copd_pen: https://iris.who.int/server/api/core/bitstreams/b9f09202-a320-4c07-ba2c-afe0d1186339/content
cvd_risk: https://iris.who.int/server/api/core/bitstreams/f106282d-01ad-4978-9a61-27fb1a8c305d/content
pneumonia: https://iris.who.int/server/api/core/bitstreams/38cf9b2a-5d7d-49de-a27b-46d5ac4fc387/content
covid19: https://iris.who.int/server/api/core/bitstreams/54fd5754-5ae5-4af1-bc1b-f15960301f17/content
malaria: https://iris.who.int/server/api/core/bitstreams/8fa903e7-5502-4c33-a300-7a3f3469bfab/content
hiv: SKIPPED (no handle, needs manual URL)
hepatitis_b: https://iris.who.int/server/api/core/bitstreams/51bfba1f-fbbe-4ae3-a950-48cf39601916/content
hepatitis_c: https://

## Complete the URL Dictionary + Map to All 36 Project Topics

In [7]:
# Manual URLs for the 4 that weren't IRIS handles
resolved_urls["diabetes"] = "https://cdn.who.int/media/docs/default-source/ncds/ncd-surveillance/guidance-on-global-monitoring-for-diabetes.pdf"
resolved_urls["obesity"] = "https://cdn.who.int/media/docs/default-source/obesity/who-discussion-paper-on-obesity---final190821.pdf"
resolved_urls["hiv"] = "https://iris.who.int/server/api/core/bitstreams/15bfbf7f-9dc6-44fe-8dc5-1beb7be8848b/content"
resolved_urls["typhoid"] = "https://iris.who.int/bitstreams/ebae84fc-9d27-420a-9e6f-41cb85873832/download"

# Map shared documents to every actual project topic (36 total) that they cover.
# Topics not listed here have no dedicated WHO guideline (confirmed gaps).
topic_to_who_doc = {
    "tuberculosis": resolved_urls["tuberculosis"],
    "hypertension": resolved_urls["hypertension"],
    "diabetes": resolved_urls["diabetes"],
    "obesity": resolved_urls["obesity"],
    "asthma": resolved_urls["asthma_copd_pen"],
    "copd": resolved_urls["asthma_copd_pen"],
    "coronary artery disease": resolved_urls["cvd_risk"],
    "heart failure": resolved_urls["cvd_risk"],
    "stroke": resolved_urls["cvd_risk"],
    "hyperlipidemia": resolved_urls["cvd_risk"],
    "pneumonia": resolved_urls["pneumonia"],
    "covid-19": resolved_urls["covid19"],
    "malaria": resolved_urls["malaria"],
    "hiv aids": resolved_urls["hiv"],
    "hepatitis b": resolved_urls["hepatitis_b"],
    "hepatitis c": resolved_urls["hepatitis_c"],
    "dengue fever": resolved_urls["dengue"],
    "typhoid": resolved_urls["typhoid"],
    "depression": resolved_urls["mhgap"],
    "anxiety disorder": resolved_urls["mhgap"],
    "epilepsy": resolved_urls["mhgap"],
    "malnutrition": resolved_urls["malnutrition"],
    "anemia in pregnancy": resolved_urls["anemia_pregnancy"],
    "breast cancer": resolved_urls["breast_cancer"],
}

print(f"Topics with WHO guidance: {len(topic_to_who_doc)}")
print(f"Topics without (confirmed gaps): {36 - len(topic_to_who_doc)}")
for topic, url in topic_to_who_doc.items():
    print(f"  {topic}: {url}")

Topics with WHO guidance: 24
Topics without (confirmed gaps): 12
  tuberculosis: https://iris.who.int/server/api/core/bitstreams/cf34aa08-c5d4-4b64-85cd-2ec1e6b14d91/content
  hypertension: https://iris.who.int/server/api/core/bitstreams/f062769d-f075-4a00-87af-0a2106e0bd04/content
  diabetes: https://cdn.who.int/media/docs/default-source/ncds/ncd-surveillance/guidance-on-global-monitoring-for-diabetes.pdf
  obesity: https://cdn.who.int/media/docs/default-source/obesity/who-discussion-paper-on-obesity---final190821.pdf
  asthma: https://iris.who.int/server/api/core/bitstreams/b9f09202-a320-4c07-ba2c-afe0d1186339/content
  copd: https://iris.who.int/server/api/core/bitstreams/b9f09202-a320-4c07-ba2c-afe0d1186339/content
  coronary artery disease: https://iris.who.int/server/api/core/bitstreams/f106282d-01ad-4978-9a61-27fb1a8c305d/content
  heart failure: https://iris.who.int/server/api/core/bitstreams/f106282d-01ad-4978-9a61-27fb1a8c305d/content
  stroke: https://iris.who.int/server/api

## Libraries

In [8]:
import pdfplumber # text and table extraction from PDFs
import fitz  # PyMuPDF's import name is 'fitz' --> image extraction from PDFs

print("pdfplumber version:", pdfplumber.__version__ if hasattr(pdfplumber, "__version__") else "installed")
print("PyMuPDF version:", fitz.version)

pdfplumber version: 0.11.4
PyMuPDF version: ('1.28.0', '1.29.0', None)


## Load the Malaria PDF (Our Test Case)

In [9]:
url = "https://iris.who.int/server/api/core/bitstreams/8fa903e7-5502-4c33-a300-7a3f3469bfab/content"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers, timeout=30)
pdf_bytes = response.content

print("Downloaded bytes:", len(pdf_bytes))

Downloaded bytes: 2852047


## Open with pdfplumber and Find That Table

In [10]:
import pdfplumber
from io import BytesIO

pdf_file = BytesIO(pdf_bytes)
plumber_pdf = pdfplumber.open(pdf_file)

print("Total pages:", len(plumber_pdf.pages))

# Search pages that contain tables in the malaria guideline PDF (451 pages)
for page_num in range(330, 345):
    page = plumber_pdf.pages[page_num]
    tables = page.find_tables()
    if tables:
        print(f"Page {page_num}: found {len(tables)} table(s)")

Total pages: 451
Page 330: found 1 table(s)
Page 331: found 1 table(s)
Page 332: found 1 table(s)
Page 333: found 1 table(s)
Page 334: found 1 table(s)
Page 335: found 1 table(s)
Page 336: found 1 table(s)
Page 338: found 1 table(s)
Page 339: found 1 table(s)
Page 340: found 1 table(s)
Page 341: found 1 table(s)
Page 342: found 1 table(s)
Page 343: found 1 table(s)
Page 344: found 1 table(s)


## Extract One Table's Actual Data

In [11]:
page = plumber_pdf.pages[330]
tables = page.extract_tables()

print(f"Number of tables on page 330: {len(tables)}")
print("---")
for row in tables[0][:10]:  # first 10 rows
    print(row)

Number of tables on page 330: 1
---
['Outcome\nTimeframe', 'Study results and\nmeasurements', 'Comparator\nNo\nintervention or\nalternative\nmedicines', 'Intervention\nSMC', 'Certainty of\nthe Evidence\n(Quality of\nevidence)', 'Summary']
['Clinical malaria:\nchildren <5\nyears, 3–4\ncycles, SP+AS\n7 Critical', 'Rate ratio 0.14\n(CI 95% 0.1 — 0.2)\nBased on data from\nparticipants in 1\nstudies. (Randomized\ncontrolled)', 'Difference:', '315 fewer per\n1000\n( CI 95% 450\nfewer — 225\nfewer )', 'High', '3–4 cycles of SMC with\nSP+AS reduces clinical\nmalaria incidence in\nchildren <5 years.']
['Clinical malaria\nincidence:\nchildren ≥5\nyears (various\nregimens)\n7 Critical', 'Rate ratio 0.27\n(CI 95% 0.25 — 0.3)\nBased on data from\nparticipants in 3\nstudies. (Randomized\ncontrolled)', 'Difference:', '170 fewer per\n1000\n( CI 95% 189\nfewer — 158\nfewer )', 'Low\nDue to serious\nrisk of bias, Due\nto serious\ninconsistency 4', 'SMC may reduce\nclinical malaria\nincidence in children

## Now Extract the Same Page's Text, Excluding the Table

In [12]:
table_obj = page.find_tables()[0]   # headings are in the first row of this table, so we can use that to extract the rest of the table data
bbox = table_obj.bbox  # (x0, top, x1, bottom)
print("Table bounding box:", bbox)     # area of the table in the page coordinates |_|

# Now exclude the table region
non_table_area = page.outside_bbox(bbox) if hasattr(page, "outside_bbox") else None
if non_table_area:
    clean_text = non_table_area.extract_text()
else:
    clean_text = "outside_bbox not available in this pdfplumber version"

print("\n=== Text WITHOUT table (new way) ===")
print(clean_text)

Table bounding box: (75.61363636363636, 76.09129090909094, 549.6622909090909, 815.0948000000001)

=== Text WITHOUT table (new way) ===
WHO guidelines for malaria - 16 October 2023 - World Health Organization (WHO)
331 of 451


## The Extraction Function to Return Both Versions

In [ ]:
import re
from typing import Optional, Tuple, List

def extract_page_content(page, header_footer_pattern: Optional[str] = None) -> Tuple[str, str, List[list]]:
    """
    Extract clean text (tables excluded), raw text (unmodified), and
    structured tables from one page.
    Returns (clean_text, raw_text, tables_list).
    """
    tables_data = []
    text_region = page

    found_tables = page.find_tables()
    for table_obj in found_tables:
        table_rows = table_obj.extract()
        tables_data.append(table_rows)

        x0, top, x1, bottom = table_obj.bbox
        x0 = max(x0, 0)
        top = max(top, 0)
        x1 = min(x1, page.width)
        bottom = min(bottom, page.height)
        safe_bbox = (x0, top, x1, bottom)

        try:
            text_region = text_region.outside_bbox(safe_bbox) if hasattr(text_region, "outside_bbox") else text_region
        except Exception:
            pass

    clean_text = text_region.extract_text() or ""
    raw_text = page.extract_text() or ""

    if header_footer_pattern:
        clean_text = re.sub(header_footer_pattern, "", clean_text).strip()
        raw_text = re.sub(header_footer_pattern, "", raw_text).strip()

    return clean_text, raw_text, tables_data

## Build the Full Document-Level Extraction Function

In [ ]:
from typing import Optional, Tuple, List

FIGURE_CAPTION_PATTERN = re.compile(r"\b(Fig(?:ure)?\.?\s*\d+|Algorithm\s*\d+)\b", re.IGNORECASE)
HEADER_FOOTER_PATTERN = r"(.*World Health Organization \(WHO\).*\n?)|(\d+ of \d+)"

def extract_full_document(pdf_bytes: bytes, header_footer_pattern: Optional[str] = HEADER_FOOTER_PATTERN):
    """
    Extract clean text, raw text, all tables, page count, and the page
    numbers of any pages whose text contains a figure/diagram/algorithm
    caption (e.g. "Fig. 1", "Algorithm 2") from an entire PDF.
    Returns (clean_text, raw_text, all_tables, num_pages, figure_pages).
    """
    pdf_file = BytesIO(pdf_bytes)
    plumber_pdf = pdfplumber.open(pdf_file)

    clean_parts = []
    raw_parts = []
    all_tables = []
    figure_pages = []

    for page_num, page in enumerate(plumber_pdf.pages):
        clean, raw, tables = extract_page_content(page, header_footer_pattern=header_footer_pattern)
        if clean:
            clean_parts.append(clean)
        if raw:
            raw_parts.append(raw)
        for table_data in tables:
            all_tables.append({"page_number": page_num, "table_data": table_data})

        if FIGURE_CAPTION_PATTERN.search(raw):
            figure_pages.append(page_num)

    num_pages = len(plumber_pdf.pages)
    return "\n".join(clean_parts), "\n".join(raw_parts), all_tables, num_pages, figure_pages


## Extract Images with PyMuPDF

In [15]:
import fitz
from typing import List, Dict, Any

def extract_images(pdf_bytes: bytes, min_width: int = 100, min_height: int = 100) -> List[Dict[str, Any]]:
    """
    Extract embedded images from a PDF, skipping tiny images (likely icons/logos).
    Returns a list of {page_number, image_index, image_bytes, ext, width, height}.
    """
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    images = []

    for page_num in range(len(doc)):
        page = doc[page_num]
        image_list = page.get_images(full=True)

        for img_index, img in enumerate(image_list):
            xref = img[0]
            base_image = doc.extract_image(xref)
            width = base_image["width"]
            height = base_image["height"]

            if width < min_width or height < min_height:
                continue

            images.append({
                "page_number": page_num,
                "image_index": img_index,
                "image_bytes": base_image["image"],
                "ext": base_image["ext"],
                "width": width,
                "height": height,
            })

    doc.close()
    return images


# Test on the same malaria PDF
extracted_images = extract_images(pdf_bytes)
print(f"Total real images found: {len(extracted_images)}")
for img in extracted_images[:5]:
    print(f"  Page {img['page_number']}: {img['width']}x{img['height']} .{img['ext']}")

Total real images found: 3
  Page 0: 1240x1754 .jpx
  Page 32: 450x277 .jpx
  Page 32: 450x297 .jpx


## Save Into the Existing data/images/ Folder and Visually Verify One Image

In [16]:
from PIL import Image
import io
from pathlib import Path

sample_image = extracted_images[1]  # one of the page 32 images

img_bytes = sample_image["image_bytes"]
pil_image = Image.open(io.BytesIO(img_bytes))

png_path = Path("../data/images/who_images_test.png")
png_path.parent.mkdir(parents=True, exist_ok=True)
pil_image.save(png_path, "PNG")

print(f"Converted and saved to {png_path.resolve()}")
print(f"Mode: {pil_image.mode}, Size: {pil_image.size}")

Converted and saved to C:\Users\DELL\Desktop\medrag\data\images\who_images_test.png
Mode: RGB, Size: (450, 277)


## Render specific pages of a PDF as full images

In [ ]:
def rasterize_figure_pages(pdf_bytes: bytes, page_numbers: List[int], dpi: int = 150) -> List[Dict[str, Any]]:
    """
    Render specific pages of a PDF as full images, to capture vector-drawn
    figures/diagrams/algorithms that have no embedded image object
    (get_images() only sees embedded bitmaps, not vector drawing instructions).
    Returns a list of {page_number, image_bytes, ext, width, height}.
    """
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    rendered = []

    for page_num in page_numbers:
        if page_num >= len(doc):
            continue
        page = doc[page_num]
        pixmap = page.get_pixmap(dpi=dpi)    # It converts a PDF page into an image --> renders (draws) the page into pixels, creating an image --> ss
        rendered.append({
            "page_number": page_num,
            "image_bytes": pixmap.tobytes("png"),
            "ext": "png",
            "width": pixmap.width,
            "height": pixmap.height,
        })

    doc.close()
    return rendered

## Dropping images duplication and not valid

In [ ]:
from collections import Counter
import hashlib
from typing import List, Dict
from traitlets import Any
from PIL import Image
import io as _io

def compute_image_hash(image_bytes: bytes) -> str:
    """Hash an image's raw bytes to detect exact duplicates (e.g. repeated logos)."""
    return hashlib.md5(image_bytes).hexdigest()


def is_blank_or_near_solid(image_bytes: bytes, std_threshold: float = 5.0) -> bool:
    """
    Detect blank/near-solid-color images (e.g. black rectangles from a
    mask/alpha-channel extraction artifact) that carry no real visual content.
    Uses pixel standard deviation as a simple, dependency-light heuristic.
    """
    try:
        img = Image.open(_io.BytesIO(image_bytes)).convert("L")  # grayscale
        pixels = list(img.getdata())
        mean = sum(pixels) / len(pixels)
        variance = sum((p - mean) ** 2 for p in pixels) / len(pixels)
        std_dev = variance ** 0.5
        return std_dev < std_threshold
    except Exception:
        return False  # if inspection fails, don't drop the image on that basis


def filter_and_dedupe_images(images: List[Dict[str, Any]], max_repeats: int = 3) -> List[Dict[str, Any]]:
    """
    Remove images that repeat more than max_repeats times across a document
    (treated as logos/branding, not real content), dedupe any remaining
    repeats down to their first occurrence, and drop blank/near-solid-color
    images (extraction artifacts with no real visual content).
    """
    images = [img for img in images if not is_blank_or_near_solid(img["image_bytes"])]

    hashes = [compute_image_hash(img["image_bytes"]) for img in images]
    hash_counts = Counter(hashes)

    seen = set()
    filtered = []
    for img, img_hash in zip(images, hashes):
        if hash_counts[img_hash] > max_repeats:
            continue  # likely a logo/branding element - drop entirely
        if img_hash not in seen:
            seen.add(img_hash)
            filtered.append(img)

    return filtered

## Data Models

In [ ]:
from pydantic import BaseModel
class Guideline(BaseModel):
    title: str
    topic: str
    clean_text: str      # tables excluded - primary field for chunking/embedding
    raw_text: str         # unmodified extraction - fallback, may contain garbled tables
    num_pages: int
    num_tables: int
    num_images: int
    source_url: str
    source: str = "who"


class WhoTable(BaseModel):
    topic: str
    page_number: int
    table_data: list  # list of rows, each row a list of cell values


class WhoImage(BaseModel):
    topic: str
    page_number: int
    image_index: int
    filename: str  # e.g. "malaria_page32_img1.png"
    width: int
    height: int
    image_type: str = "embedded"

## Save to disk function

In [ ]:
from PIL import Image
import io as _io

def save_who_guideline(guideline: Guideline, output_dir: str) -> Path:
    """Write a WHO guideline's text to data/raw/who/{topic}.json."""
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    filepath = Path(output_dir) / f"{guideline.topic.replace(' ', '_')}.json"

    with open(filepath, "w", encoding="utf-8") as f:
        f.write(guideline.model_dump_json())

    return filepath

def save_who_tables(tables: List[dict], topic: str, output_dir: str) -> Path:
    """
    Write tables extracted from a WHO guideline to data/tables/who/{topic}.jsonl.
    tables: list of {page_number, table_data} dicts from extract_full_document.
    """
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    filepath = Path(output_dir) / f"{topic.replace(' ', '_')}.jsonl"

    with open(filepath, "w", encoding="utf-8") as f:
        for t in tables:
            record = WhoTable(topic=topic, page_number=t["page_number"], table_data=t["table_data"])
            f.write(record.model_dump_json() + "\n")

    return filepath

def save_who_images(images: List[dict], topic: str, output_dir: str) -> List[WhoImage]:
    """
    Save extracted images as PNG files to data/images/who/, plus a metadata
    JSONL file. images: list of dicts from extract_images/rasterize_figure_pages
    (with raw image_bytes and an image_type field). Returns the saved WhoImage
    records.
    """

    Path(output_dir).mkdir(parents=True, exist_ok=True)
    saved_records = []

    for i, img in enumerate(images):
        pil_image = Image.open(_io.BytesIO(img["image_bytes"]))
        filename = f"{topic.replace(' ', '_')}_page{img['page_number']}_img{i}.png"
        filepath = Path(output_dir) / filename
        pil_image.save(filepath, "PNG")

        record = WhoImage(
            topic=topic,
            page_number=img["page_number"],
            image_index=i,
            filename=filename,
            width=img["width"],
            height=img["height"],
            image_type=img.get("image_type", "embedded"),
        )
        saved_records.append(record)

    meta_path = Path(output_dir) / f"{topic.replace(' ', '_')}_metadata.jsonl"
    with open(meta_path, "w", encoding="utf-8") as f:
        for record in saved_records:
            f.write(record.model_dump_json() + "\n")

    return saved_records

## Full per-topic pipeline

In [ ]:
BROWSER_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
}

def fetch_who_guideline(topic: str, url: str, title: str, headers: dict = BROWSER_HEADERS) -> Tuple[Guideline, List[dict], List[dict]]:
    """
    Download a WHO guideline PDF and extract text, tables, and images in one pass.
    Images include both real embedded images (PyMuPDF get_images) and
    rasterized full-page renders for any page whose text contains a
    figure/diagram/algorithm caption but has no embedded image object
    (vector-drawn content that get_images() cannot see).
    Returns (Guideline, tables, images) - tables/images are the raw extraction
    results (not yet converted to WhoTable/WhoImage records or saved to disk).
    """
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    pdf_bytes = response.content

    clean_text, raw_text, tables, num_pages, figure_pages = extract_full_document(pdf_bytes)

    embedded_images = extract_images(pdf_bytes)
    embedded_images = filter_and_dedupe_images(embedded_images)
    for img in embedded_images:
        img["image_type"] = "embedded"

    # Only rasterize figure-caption pages that didn't already yield a real
    # embedded image, to avoid redundant near-duplicate content
    pages_with_embedded = {img["page_number"] for img in embedded_images} # pages that already have an embedded image, so we don't need to rasterize them again
    pages_to_rasterize = [p for p in figure_pages if p not in pages_with_embedded]

    rasterized_images = rasterize_figure_pages(pdf_bytes, pages_to_rasterize)
    rasterized_images = [img for img in rasterized_images if not is_blank_or_near_solid(img["image_bytes"])]
    for img in rasterized_images:
        img["image_type"] = "rasterized_page"

    all_images = embedded_images + rasterized_images

    guideline = Guideline(
        title=title,
        topic=topic,
        clean_text=clean_text,
        raw_text=raw_text,
        num_pages=num_pages,
        num_tables=len(tables),
        num_images=len(all_images),
        source_url=url,
    )

    return guideline, tables, all_images

## Full Batch Run — All 17 Documents, 24 Topics

In [21]:
import time
import os

# topic -> (url, title) - reuse from earlier resolution work
# Several topics intentionally share the same source document
TOPIC_DOCS = {
    "tuberculosis": ("https://iris.who.int/server/api/core/bitstreams/cf34aa08-c5d4-4b64-85cd-2ec1e6b14d91/content", "WHO consolidated guidelines on tuberculosis: Module 4 Treatment"),
    "hypertension": ("https://iris.who.int/server/api/core/bitstreams/f062769d-f075-4a00-87af-0a2106e0bd04/content", "Guideline for the pharmacological treatment of hypertension in adults"),
    "diabetes": ("https://cdn.who.int/media/docs/default-source/ncds/ncd-surveillance/guidance-on-global-monitoring-for-diabetes.pdf", "Guidance on global monitoring for diabetes prevention and control"),
    "obesity": ("https://cdn.who.int/media/docs/default-source/obesity/who-discussion-paper-on-obesity---final190821.pdf", "WHO discussion paper on obesity"),
    "asthma": ("https://iris.who.int/server/api/core/bitstreams/b9f09202-a320-4c07-ba2c-afe0d1186339/content", "WHO package of essential noncommunicable (PEN) disease interventions"),
    "copd": ("https://iris.who.int/server/api/core/bitstreams/b9f09202-a320-4c07-ba2c-afe0d1186339/content", "WHO package of essential noncommunicable (PEN) disease interventions"),
    "coronary artery disease": ("https://iris.who.int/server/api/core/bitstreams/f106282d-01ad-4978-9a61-27fb1a8c305d/content", "Prevention of cardiovascular disease: guidelines for assessment and management of total cardiovascular risk"),
    "heart failure": ("https://iris.who.int/server/api/core/bitstreams/f106282d-01ad-4978-9a61-27fb1a8c305d/content", "Prevention of cardiovascular disease: guidelines for assessment and management of total cardiovascular risk"),
    "stroke": ("https://iris.who.int/server/api/core/bitstreams/f106282d-01ad-4978-9a61-27fb1a8c305d/content", "Prevention of cardiovascular disease: guidelines for assessment and management of total cardiovascular risk"),
    "hyperlipidemia": ("https://iris.who.int/server/api/core/bitstreams/f106282d-01ad-4978-9a61-27fb1a8c305d/content", "Prevention of cardiovascular disease: guidelines for assessment and management of total cardiovascular risk"),
    "pneumonia": ("https://iris.who.int/server/api/core/bitstreams/38cf9b2a-5d7d-49de-a27b-46d5ac4fc387/content", "Revised WHO classification and treatment of childhood pneumonia at health facilities"),
    "covid-19": ("https://iris.who.int/server/api/core/bitstreams/54fd5754-5ae5-4af1-bc1b-f15960301f17/content", "Clinical management of COVID-19: living guideline"),
    "malaria": ("https://iris.who.int/server/api/core/bitstreams/8fa903e7-5502-4c33-a300-7a3f3469bfab/content", "WHO guidelines for malaria"),
    "hiv aids": ("https://iris.who.int/server/api/core/bitstreams/15bfbf7f-9dc6-44fe-8dc5-1beb7be8848b/content", "WHO updated recommendations on HIV clinical management"),
    "hepatitis b": ("https://iris.who.int/server/api/core/bitstreams/51bfba1f-fbbe-4ae3-a950-48cf39601916/content", "Guidelines for the prevention, care and treatment of persons with chronic hepatitis B infection"),
    "hepatitis c": ("https://iris.who.int/server/api/core/bitstreams/8bad8f1b-84c4-4abd-9653-99dcc807e6ff/content", "Guidelines for the care and treatment of persons diagnosed with chronic hepatitis C virus infection"),
    "dengue fever": ("https://iris.who.int/server/api/core/bitstreams/825eb07b-b372-4527-ac2f-fe6e8201c1c1/content", "Handbook for clinical management of dengue"),
    "typhoid": ("https://iris.who.int/server/api/core/bitstreams/ebae84fc-9d27-420a-9e6f-41cb85873832/content", "Background document: the diagnosis, treatment and prevention of typhoid fever"),
    "depression": ("https://iris.who.int/server/api/core/bitstreams/6b9d19fe-b732-4065-bd2f-9736f4061a7e/content", "mhGAP guideline for mental, neurological and substance use disorders"),
    "anxiety disorder": ("https://iris.who.int/server/api/core/bitstreams/6b9d19fe-b732-4065-bd2f-9736f4061a7e/content", "mhGAP guideline for mental, neurological and substance use disorders"),
    "epilepsy": ("https://iris.who.int/server/api/core/bitstreams/6b9d19fe-b732-4065-bd2f-9736f4061a7e/content", "mhGAP guideline for mental, neurological and substance use disorders"),
    "malnutrition": ("https://iris.who.int/server/api/core/bitstreams/0c5e1662-a300-4827-90fa-962ac21360dd/content", "Guideline: updates on the management of severe acute malnutrition in infants and children"),
    "anemia in pregnancy": ("https://iris.who.int/server/api/core/bitstreams/f9f74397-1440-478d-a63c-26f29a01552f/content", "WHO guideline on anaemia"),
    "breast cancer": ("https://iris.who.int/server/api/core/bitstreams/efbe53e5-9354-482d-8c75-94cc7a1110b4/content", "WHO position paper on mammography screening"),
}

GUIDELINE_DIR = os.path.abspath(os.path.join(os.path.dirname(__file__), "..", "..", "data", "raw", "who"))
TABLE_DIR = os.path.abspath(os.path.join(os.path.dirname(__file__), "..", "..", "data", "tables", "who"))
IMAGE_DIR = os.path.abspath(os.path.join(os.path.dirname(__file__), "..", "..", "data", "images", "who"))


processed_urls = {}  # cache: same URL processed only once, even if multiple topics share it
results = []

for topic, (url, title) in TOPIC_DOCS.items():
    if url in processed_urls:
        cached = processed_urls[url]
        save_who_guideline(
            cached["guideline"].model_copy(update={"topic": topic}),
            output_dir=GUIDELINE_DIR,
        )
        save_who_tables(cached["tables"], topic=topic, output_dir=TABLE_DIR)
        save_who_images(cached["images"], topic=topic, output_dir=IMAGE_DIR)
        print(f"{topic}: reused cached extraction from shared document")
        results.append({"topic": topic, "status": "success (cached)"})
        continue

    try:
        guideline, tables, images = fetch_who_guideline(
            topic=topic, url=url, title=title, headers=BROWSER_HEADERS
        )

        save_who_guideline(guideline, output_dir=GUIDELINE_DIR)
        save_who_tables(tables, topic=topic, output_dir=TABLE_DIR)
        save_who_images(images, topic=topic, output_dir=IMAGE_DIR)

        processed_urls[url] = {"guideline": guideline, "tables": tables, "images": images}
        print(
            f"{topic}: saved ({guideline.num_pages} pages, {guideline.num_tables} tables, "
            f"{guideline.num_images} images)"
        )
        results.append({"topic": topic, "status": "success"})

    except Exception as e:
        print(f"{topic}: FAILED - {e}")
        results.append({"topic": topic, "status": "failed", "error": str(e)})

    time.sleep(1)  # be polite to WHO's servers between downloads

success_count = sum(1 for r in results if "success" in r["status"])
print(f"=== DONE === {success_count}/{len(results)} succeeded")

tuberculosis: saved (72 pages, 14 tables, 0 images)
hypertension: saved (61 pages, 33 tables, 2 images)
diabetes: saved (94 pages, 117 tables, 1 images)
obesity: saved (16 pages, 2 tables, 1 images)
asthma: saved (85 pages, 74 tables, 3 images)
copd: reused cached extraction from shared document
coronary artery disease: saved (92 pages, 171 tables, 0 images)
heart failure: reused cached extraction from shared document
stroke: reused cached extraction from shared document
hyperlipidemia: reused cached extraction from shared document
pneumonia: saved (34 pages, 1 tables, 4 images)
covid-19: saved (182 pages, 89 tables, 4 images)
malaria: saved (451 pages, 275 tables, 3 images)
hiv aids: saved (108 pages, 54 tables, 0 images)
hepatitis b: saved (166 pages, 46 tables, 2 images)
hepatitis c: saved (14 pages, 2 tables, 3 images)
dengue fever: saved (124 pages, 50 tables, 32 images)
typhoid: saved (48 pages, 7 tables, 2 images)
depression: saved (71 pages, 69 tables, 3 images)
anxiety disorde